In [2]:
import pandas as pd

# Load the dataset
file_path = "megaGymDataset.csv"
df = pd.read_csv(file_path)

# Display basic info and first few rows
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2918 entries, 0 to 2917
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  2918 non-null   int64  
 1   Title       2918 non-null   object 
 2   Desc        1368 non-null   object 
 3   Type        2918 non-null   object 
 4   BodyPart    2918 non-null   object 
 5   Equipment   2886 non-null   object 
 6   Level       2918 non-null   object 
 7   Rating      1031 non-null   float64
 8   RatingDesc  862 non-null    object 
dtypes: float64(1), int64(1), object(7)
memory usage: 205.3+ KB


,Unnamed: 0,Title,Desc,Type,BodyPart,Equipment,Level,Rating,RatingDesc
0,0,Partner plank band row,The partner plank band row is an abdominal exe...,Strength,Abdominals,Bands,Intermediate,0.0,NaN
1,1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
2,2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
3,3,Banded crunch,The banded crunch is an exercise targeting the...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
4,4,Crunch,The crunch is a popular core exercise targetin...,Strength,Abdominals,Bands,Intermediate,NaN,NaN


In [6]:
# Select only numeric columns for mean imputation
numeric_cols = df.select_dtypes(include=["number"]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())


In [8]:
print(df.columns)


Index(['Unnamed: 0', 'Title', 'Desc', 'Type', 'BodyPart', 'Equipment', 'Level',
       'Rating', 'RatingDesc'],
      dtype='object')


In [9]:
num_features = ["Age", "Weight", "Height", "Max_BPM", "Avg_BPM", "Resting_BPM", 
                "Session_Duration", "Calories_Burned", "Fat_Percentage", "Water_Intake"]

# Check which features are missing
missing_features = [col for col in num_features if col not in df.columns]
print("Missing features:", missing_features)


Missing features: ['Age', 'Weight', 'Height', 'Max_BPM', 'Avg_BPM', 'Resting_BPM', 'Session_Duration', 'Calories_Burned', 'Fat_Percentage', 'Water_Intake']


In [11]:
print(df[num_features].dtypes)


Series([], dtype: object)


In [12]:
# Ensure all selected features are numeric
df[num_features] = df[num_features].apply(pd.to_numeric, errors="coerce")

# Check again if there are still missing values
print(df[num_features].isnull().sum())


Series([], dtype: float64)


In [13]:
df[num_features] = df[num_features].fillna(df[num_features].mean())


In [15]:
print("Selected Features:", num_features)


Selected Features: []


In [16]:
df[num_features] = df[num_features].apply(pd.to_numeric, errors="coerce")


In [17]:
print(df[num_features].isnull().sum())


Series([], dtype: float64)


In [18]:
from sklearn.preprocessing import StandardScaler

# Make sure we still have numeric columns left
if len(num_features) > 0:
    scaler = StandardScaler()
    df[num_features] = scaler.fit_transform(df[num_features])
    print("✅ Scaling applied successfully!")
else:
    print("❌ No numeric columns found for scaling!")


❌ No numeric columns found for scaling!


In [20]:
print(df.columns)


Index(['Unnamed: 0', 'Title', 'Desc', 'Type', 'BodyPart', 'Equipment', 'Level',
       'Rating', 'RatingDesc'],
      dtype='object')


In [21]:
df.columns = df.columns.str.strip()  # Remove spaces
df.rename(columns={"workout_frequency": "Workout_Frequency"}, inplace=True)  # Fix naming


In [23]:
print("Columns in dataset:", df.columns)


Columns in dataset: Index(['Unnamed: 0', 'Title', 'Desc', 'Type', 'BodyPart', 'Equipment', 'Level',
       'Rating', 'RatingDesc'],
      dtype='object')


In [24]:
# Drop unnecessary columns
X = df.drop(columns=["Unnamed: 0", "Rating"])  # Features
y = df["Rating"]  # Target variable


In [25]:
from sklearn.preprocessing import LabelEncoder

# Encode all categorical columns
categorical_cols = ["Title", "Desc", "Type", "BodyPart", "Equipment", "Level", "RatingDesc"]
for col in categorical_cols:
    X[col] = LabelEncoder().fit_transform(X[col])

# Check transformed data
X.head()


,Title,Desc,Type,BodyPart,Equipment,Level,RatingDesc
0,2078,696,4,0,0,2,1
1,323,139,4,0,0,2,1
2,803,143,4,0,0,2,1
3,322,138,4,0,0,2,1
4,590,298,4,0,0,2,1


In [26]:
from sklearn.model_selection import train_test_split

# Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"✅ Training samples: {X_train.shape[0]}, Testing samples: {X_test.shape[0]}")


✅ Training samples: 2334, Testing samples: 584


In [27]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Define models
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=10, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
}

# Train and evaluate each model
for name, model in models.items():
    model.fit(X_train, y_train)  # Train model
    y_pred = model.predict(X_test)  # Predict ratings
    
    # Print evaluation metrics
    print(f"\n🔹 Model: {name}")
    print(f"Mean Absolute Error: {mean_absolute_error(y_test, y_pred):.2f}")
    print(f"Mean Squared Error: {mean_squared_error(y_test, y_pred):.2f}")
    print(f"R² Score: {r2_score(y_test, y_pred):.2f}")



🔹 Model: Linear Regression
Mean Absolute Error: 1.00
Mean Squared Error: 3.15
R² Score: 0.27

🔹 Model: Decision Tree
Mean Absolute Error: 0.66
Mean Squared Error: 3.22
R² Score: 0.25

🔹 Model: Random Forest
Mean Absolute Error: 0.61
Mean Squared Error: 2.36
R² Score: 0.45


In [28]:
import joblib

# Save the best model
joblib.dump(models["Random Forest"], "rating_predictor.pkl")

print("✅ Model saved as 'rating_predictor.pkl'")


✅ Model saved as 'rating_predictor.pkl'


In [30]:
from sklearn.preprocessing import LabelEncoder

# Store label encoders for all categorical columns
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])  # Transform training data
    encoders[col] = le  # Save encoder for later use

# Check transformed data
X.head()


,Title,Desc,Type,BodyPart,Equipment,Level,RatingDesc
0,2078,696,4,0,0,2,1
1,323,139,4,0,0,2,1
2,803,143,4,0,0,2,1
3,322,138,4,0,0,2,1
4,590,298,4,0,0,2,1


In [31]:
# Sample new exercise
new_exercise = pd.DataFrame([{
    "Title": "Push Ups", "Desc": "Chest workout", "Type": "Strength", "BodyPart": "Chest",
    "Equipment": "None", "Level": "Beginner", "RatingDesc": "Good"
}])

# Transform new data using the same encoders
for col in categorical_cols:
    if col in new_exercise.columns:
        if new_exercise[col][0] in encoders[col].classes_:
            new_exercise[col] = encoders[col].transform(new_exercise[col])
        else:
            new_exercise[col] = -1  # Assign unseen values to a new category (-1)

# Predict rating
predicted_rating = loaded_model.predict(new_exercise)
print(f"Predicted Rating: {predicted_rating[0]:.2f}")


Predicted Rating: 5.75
